# 02: MONAI 3D U-Net Pipeline & Interactive Segmentation Dashboard

In this notebook, we build, train, and interactively evaluate a **3D Deep Learning Segmentation Model** for multi-modal brain MRI:
1. **MONAI Preprocessing Pipeline**: `Orientationd(RAS)`, `Spacingd(1mm)`, `NormalizeIntensityd`, and `RandCropByPosNegLabeld`.
2. **Interactive 3D Training Patch Explorer**: Inspect augmented 3D sub-volumes ($96 \times 96 \times 96$) sampled around tumors.
3. **Custom 3D U-Net Architecture**: 3D convolutions with residual blocks, instance normalization, and multi-scale skip connections.
4. **Training Loop with DiceCELoss**: Handling severe class imbalance.
5. **Interactive Full-Volume Inference & Evaluation Dashboard**: Side-by-side interactive comparison of MRI scans, Ground Truth masks, and 3D U-Net Model Predictions with live slice Dice score metrics.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.dataset import get_dataloaders
from src.model import get_model
from src.train import run_training
from src.visualization import (
    plot_orthogonal_slices,
    interactive_orthogonal_viewer,
    interactive_segmentation_comparison,
    plot_3d_tumor_mesh_plotly,
    BRATS_LABEL_NAMES
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

---
## 1. MONAI DataLoaders & Dataset Pipeline
We load the BraTS training and validation datasets with MONAI's dictionary transform pipelines.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "Task01_BrainTumour"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train_loader, val_loader, metadata = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=2,
    roi_size=(96, 96, 96),
    val_ratio=0.2,
    num_samples_per_volume=2,
    max_samples=25
)

print(f"Dataset Modalities : {metadata.get('modality')}")
print(f"Segmentation Labels: {metadata.get('labels')}")
print(f"Training Batches   : {len(train_loader)} | Validation Batches: {len(val_loader)}")

---
## 2. Interactive 3D Training Patch Browser
`RandCropByPosNegLabeld` extracts 3D sub-volume patches ($96 \times 96 \times 96$) balanced between foreground tumor regions and background brain tissue.

Use the interactive viewer below to explore an augmented 3D training patch across all 3 orthogonal planes:

In [ ]:
# Fetch a training batch
sample_batch = next(iter(train_loader))
patch_images = sample_batch["image"].numpy()  # (B, 4, 96, 96, 96)
patch_labels = sample_batch["label"].numpy()  # (B, 1, 96, 96, 96)

print(f"Patch Batch Image Shape: {patch_images.shape} (Batch, Modalities, X, Y, Z)")
print(f"Patch Batch Label Shape: {patch_labels.shape} (Batch, 1, X, Y, Z)")

# Interactive 3D Patch Viewer
interactive_orthogonal_viewer(
    volume=patch_images[0],
    mask=patch_labels[0, 0],
    modality_names=["0: FLAIR", "1: T1", "2: T1ce", "3: T2"]
)

---
## 3. 3D U-Net Model Architecture
We instantiate the custom modular 3D U-Net with 3D convolutions, residual blocks, instance normalization, and multi-scale skip connections.

In [ ]:
model = get_model("custom_unet", in_channels=4, out_channels=4).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"3D U-Net initialized with {total_params:,} trainable parameters.")

---
## 4. Model Training & Checkpoint Loading
You can train the model or load the pre-trained best checkpoint to run full-volume evaluation immediately.

In [ ]:
checkpoint_path = RESULTS_DIR / "best_metric_model.pth"

# If checkpoint doesn't exist, run a training run
if not checkpoint_path.exists():
    print("[>] Running 3D U-Net training pipeline...")
    run_training(
        data_dir=DATA_DIR,
        output_dir=RESULTS_DIR,
        model_name="custom_unet",
        epochs=10,
        batch_size=2,
        lr=2e-4,
        roi_size=(96, 96, 96),
        sw_batch_size=2,
        val_interval=1,
        max_samples=20
    )
else:
    print(f"[+] Found existing trained model checkpoint at: {checkpoint_path}")

# Load best checkpoint into model
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
print("[+] Model loaded and ready for inference!")

---
## 5. Interactive Whole-Volume Inference & Segmentation Dashboard
Using `sliding_window_inference`, we evaluate whole-brain 3D volumes without GPU out-of-memory errors.

**Interactive Dashboard:**
* **Axial Slice Slider**: Scroll through all cross-sectional slices of the patient's brain.
* **Side-by-Side Panels**:
  1. **Structural MRI Scan** (T1ce / Contrast-Enhanced)
  2. **Ground Truth Annotation** (Expert Neuroradiologist label)
  3. **3D U-Net Model Prediction** (with real-time slice-level Dice score calculation!)

In [ ]:
from monai.inferers import sliding_window_inference

# Run sliding window inference on first validation subject
val_batch = next(iter(val_loader))
val_image = val_batch["image"].to(device)
val_label = val_batch["label"][0, 0].cpu().numpy()

print(f"Running Whole-Volume 3D Inference on Volume shape: {val_image.shape}...")
with torch.no_grad():
    val_output = sliding_window_inference(
        inputs=val_image,
        roi_size=(96, 96, 96),
        sw_batch_size=2,
        predictor=model,
        overlap=0.5
    )
    pred_mask = torch.argmax(val_output, dim=1)[0].cpu().numpy()

val_img_np = val_image[0].cpu().numpy()

# Calculate overall whole-volume Dice score
gt_fg = (val_label > 0)
pred_fg = (pred_mask > 0)
overall_dice = (2.0 * np.logical_and(gt_fg, pred_fg).sum()) / (gt_fg.sum() + pred_fg.sum() + 1e-8)
print(f"[★] Whole-Volume Foreground Dice Score: {overall_dice:.4f}")

# Launch interactive 3-panel segmentation dashboard
interactive_segmentation_comparison(
    image_4d=val_img_np,
    gt_mask=val_label,
    pred_mask=pred_mask,
    modality_idx=2  # T1ce
)

---
## 6. Interactive 3D Model Prediction Surface (Plotly)
Compare the 3D surface geometry of the **Model Prediction** against the **Ground Truth** in interactive 3D:

In [ ]:
fig_pred_3d = plot_3d_tumor_mesh_plotly(
    mask=pred_mask,
    step_size=2,
    title="3D U-Net Predicted Tumor Volume (Interactive 3D Surface)"
)
if fig_pred_3d:
    fig_pred_3d.show()

---
### Project 2 Summary & Transition to Project 3
* **What you built**: An end-to-end 3D medical volumetric pipeline covering raw NIfTI loading, affine coordinate math, MONAI 3D patch augmentation, 3D U-Net residual architecture, and sliding-window full volume evaluation.
* **Why this prepares you for Project 3**: In diffusion MRI and tractography, 3D fiber streamlines originate directly from these volumetric coordinate spaces. Now you are ready to tackle **Synthetic & Real Fiber Classification with Sequence EdgeConvolutions** in Project 3!